In [1]:
from cropspectai import ml_config
from cropspectai.data_analysis import preprocessing
from cropspectai.data_analysis.exploration import explore_dataset
from cropspectai.modeling.train import train_model

import pandas as pd

from sklearn.model_selection import train_test_split
import lightgbm as lgb 
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.cross_decomposition import PLSRegression

## load data

In [2]:
train_path = ml_config.PROCESSED_PARQUET_DATA_SPLITS_PATH / "all" / "train.parquet"
df = pd.read_parquet(train_path)

print(df.columns)
print(df.shape)

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'cab', 'cw', 'cm', 'LAI', 'cp', 'cbc', 'car', 'anth'],
      dtype='str')
(3550, 20)


## DROP duplicates based on Nan in target

In [3]:
target = "cab"

df = preprocessing.drop_duplicates(df, target).copy()

(Before) Shape of dataframe: (3550, 20)
Dropping duplicate rows
Duplicate rows: 0
(After) Shape of dataframe: (3550, 20)
Checking for rows without labels
Rows with no labels: 1943
(After) Shape of dataframe: (1607, 13)
Target columns dropped too


In [4]:
X, y = preprocessing.split_dataset_into_X_y(df, target) # pass target when df only has that one target column

In [7]:
explore_dataset(X)

Shape
Rows: 1,607
Columns: 12
Data Types
F1     float64
F2     float64
FZ     float64
F3     float64
F4     float64
FY     float64
F5     float64
FXL    float64
F6     float64
F7     float64
F8     float64
NIR    float64
dtype: object
Missing Values
Empty DataFrame
Columns: [Missing, Percent]
Index: []
Duplicate Rows
Duplicates:              6
Percent:                 0.4%
Duplicates all NaN:      0
Statistical Summary
                 F1            F2           FZ           F3           F4  \
count  1.607000e+03  1.607000e+03  1607.000000  1607.000000  1607.000000   
mean   1.908424e-02  2.026702e-02     0.022770     0.025308     0.041101   
std    1.173288e-02  1.229533e-02     0.013661     0.015190     0.019335   
min    1.507759e-14  3.092801e-15     0.000044     0.000078     0.002931   
25%    1.244905e-02  1.370406e-02     0.015433     0.016607     0.028211   
50%    1.836949e-02  1.909791e-02     0.021002     0.022945     0.037121   
75%    2.395043e-02  2.431003e-02     0.02671

In [8]:
explore_dataset(y)

Shape
Rows: 1,607
Columns: 1
Data Types
cab    float64
dtype: object
Missing Values
Empty DataFrame
Columns: [Missing, Percent]
Index: []
Duplicate Rows
Duplicates:              166
Percent:                 10.3%
Duplicates all NaN:      0
Statistical Summary
               cab
count  1607.000000
mean     37.887584
std      15.145647
min       5.164474
25%      26.772733
50%      35.935786
75%      48.275792
max     229.497477


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

In [ ]:
explore_dataset(y_train)

In [ ]:
explore_dataset(y_val)

## transform - impute

In [ ]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()
print(numerical_columns)

In [ ]:
categorical_columns = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
print(categorical_columns)

## XGBoost
XGBoost supports multi-output but default multi-output strategy is essentially one model per target. Therefore, it does not learn relationships between targets.

In [ ]:
model_xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=1.0,

    tree_method="hist",

    objective="reg:squarederror",
    eval_metric="rmse",
    # early_stopping_rounds=50,

    n_jobs=-1,
    random_state=42
)

In [ ]:
pipeline_xgb = train_model(
    model_xgb,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=False
)

## light gbm

In [ ]:
model_lgb = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,

    num_leaves=15,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    objective="regression",

    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

In [ ]:
pipeline_lgb = train_model(
    model_lgb,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=False
)

## Random forest

In [ ]:
model_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42,
)

In [ ]:
pipeline_rf = train_model(
    model_rf,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=False
)

## linear models with regularization

In [ ]:
model_ridge = Ridge(
    alpha=0.01,
    max_iter=10000,
)

In [ ]:
pipeline_ridge = train_model(
    model_ridge,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=True
)

In [ ]:
model_lasso = Lasso(
    alpha=0.01,
    max_iter=10000,
    random_state=42,
)

In [ ]:
pipeline_lasso = train_model(
    model_lasso,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=True
)

## PLS regression

In [ ]:
model_pls = PLSRegression(
    n_components=5,
    scale=False, # set it false to avoid scaling twice
    max_iter=500,
    tol=1e-06,
)

In [ ]:
pipeline_pls = train_model(
    model_pls,
    X_train,
    y_train,
    X_val,
    y_val,
    scale=True
)